# Module 2 · Demo — Talking to the Model

**From 0 to Agentic AI — DataHack Summit 2026**

A plain LLM is *prompt → response*. Powerful, but on its own it hits three walls:
it **hallucinates**, it **can't act**, and it **forgets**. In this demo we feel those walls,
then take the first two steps past them — **structured outputs** and **tool calling** —
and finish by pressing the *easy button*: a working agent in ~5 lines.

### What you'll learn
1. Make a plain LLM call with LangChain (`invoke`, `stream`)
2. Watch the three walls fail, live
3. Turn free text into trustworthy **structured output** (Pydantic)
4. **Tool calling**: the model *requests* a function; a prebuilt agent *runs* it

---
## Setup

In [ ]:
# Install the workshop stack (Colab). Locally, use `uv sync` instead.
# Version ranges match src/pyproject.toml (the single source of truth).
!pip install -q "langchain>=1.2,<2" "langchain-openai>=1.1,<2" \
               "langgraph>=1.0,<2" "langchain-tavily>=0.2"

In [ ]:
import os
from getpass import getpass

# Local: load keys from src/.env (walks up from this notebook to find it).
# Colab: no .env, so you'll be prompted for any missing key.
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    pass

for key in ["OPENAI_API_KEY"]:
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")

---
## Part 1 · A plain LLM call

`init_chat_model` gives one uniform interface across providers. A chat model takes a list of
messages — a **system** message (rules/persona) and a **human** message (the task).

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4.1-mini", model_provider="openai", temperature=0)

resp = llm.invoke([
    ("system", "You are a concise assistant for a software company."),
    ("human", "In one sentence, what is an AI agent?"),
])
print(resp.content)

### Streaming — token by token
Same call, but stream the tokens as they arrive (the UX we'll reuse in the Streamlit app).

In [ ]:
for chunk in llm.stream("List three uses of AI agents in a company, very briefly."):
    print(chunk.content, end="", flush=True)

---
## Part 2 · Watch it hit the walls

Point the bare model at three real company-ops questions. Each fails differently.

### 🧠 Wall 1 — No knowledge of your world
The model has never seen your HR policy. So it either **admits it can't know** (a well-aligned
model, below) or — more dangerously — **invents** a confident answer. Neither is usable.

In [ ]:
resp = llm.invoke("How many paid time-off (PTO) days do employees at our company get?")
print(resp.content)

# ⬆️ It has no access to YOUR data, so it can't answer reliably — it can only refuse or guess.
#    This is the wall RAG breaks (Module 6): give the model your documents to read.

### 🖐️ Wall 2 — Can't act
Ask it to *do* something in the world.

In [ ]:
resp = llm.invoke("Send a Slack message to the billing team about the failing deploy.")
print(resp.content)

# ⬆️ It writes a lovely message — but it cannot actually send anything. Just text.

### 🔁 Wall 3 — No memory
Each `invoke` is independent. The model can't remember what we said a moment ago.

In [ ]:
llm.invoke("My name is Alessandro and I work on the billing service.")
resp = llm.invoke("What service do I work on?")
print(resp.content)

# ⬆️ Blank stare: the second call knows nothing about the first (we never passed the history).

> **The gaps map to the fixes.** No knowledge → give it your data (RAG). Can't act → give it *tools*. Forgets → give it *state*. The rest of the workshop closes these three walls, one at a time. First: tools.

---
## Part 3 · Structured outputs

Before tools, we need the model's answers to be **machine-readable**. Ask for a Pydantic
object instead of prose, and code downstream can *trust* the shape.

In [ ]:
from pydantic import BaseModel, Field

class PolicyAnswer(BaseModel):
    """A structured answer to a company-policy question."""
    answer: str = Field(description="the direct answer, or empty if unknown")
    confidence: str = Field(description="one of: high, medium, low")
    needs_lookup: bool = Field(
        description="true if answering reliably requires reading an internal "
                    "company document the model has not been given")

structured_llm = llm.with_structured_output(PolicyAnswer)
result = structured_llm.invoke(
    "How many PTO days do employees at OUR company get?")
print(type(result))
print(result)

Now `result.needs_lookup` is a real boolean we can branch on — no string parsing. For an
*internal* policy question it should come back `True`: the model knows it needs a document
it doesn't have. That flag is exactly the kind of signal that drives **routing** later —
and it's the backbone of tool-calling too.

---
## Part 4 · Tool calling

A tool is a function you *describe* to the model. It replies with a **request** to call one
— it never runs code itself. First, see the raw request.

In [ ]:
from langchain_core.tools import tool

@tool
def get_hr_policy(topic: str) -> str:
    """Look up the company HR policy for a given topic (e.g. 'PTO', 'remote work')."""
    # Canned data for the demo — in the real app this hits a real source.
    faq = {"pto": "Full-time employees get 28 PTO days per year.",
           "remote": "Remote work is allowed up to 3 days per week."}
    return faq.get(topic.lower().strip(), "No policy found for that topic.")

llm_with_tools = llm.bind_tools([get_hr_policy])
resp = llm_with_tools.invoke("How many PTO days do we get?")

print("Text content:", repr(resp.content))       # usually empty
print("Tool calls: ", resp.tool_calls)            # the {name, args} request

The model didn't answer — it **asked** us to call `get_hr_policy(topic='PTO')`.
Running that tool and feeding the result back is one turn of the *agent loop*.

We could wire that loop by hand (we will, in Module 4). For now — the **easy button**.

### The easy button — a prebuilt agent
`create_agent` wires the model to the tools and runs the whole
*reason → act → observe* loop for us. (It builds a **LangGraph** graph under the hood —
the very loop we'll rebuild by hand in Module 4.)

In [ ]:
from langchain.agents import create_agent

agent = create_agent(llm, tools=[get_hr_policy])

result = agent.invoke({"messages": [("user", "How many PTO days do we get?")]})

# The final answer:
print(result["messages"][-1].content)

### Look inside — the message trace
The magic is just messages. Print each one to see: the tool **request**, the tool **result**,
then the model's **final answer**.

In [ ]:
for m in result['messages']:
    m.pretty_print()

---
## Key takeaways
- A bare LLM **hallucinates, can't act, and forgets** — three walls.
- **Structured outputs** (Pydantic) turn text into data your code can trust.
- **Tool calling** = the model *requests* a function; your runtime *runs* it. Safe by design.
- `create_agent` gives you the whole loop prebuilt — a real agent in ~5 lines.

➡️ **Next demo:** point a prebuilt agent at a *real* tool — web search — and answer a
company question end-to-end. Then Module 3 opens the loop up; the real app build starts in Module 4.